# Day 2 Starter Kit: Complete the Grounding

**Your dataset: Payment, shipment & invoice (code PAY).** An exporter's finance and logistics log: invoices, payments and disputes, collections, bookings, freight tracking, customs, tax and credit.

**The idea.** Retrieval finds records that *sound* similar. It cannot tell whether two records describe the *same event*. The graph can, **if it knows which parts of a record identify an event.** That knowledge is a list of regular expressions, and writing it well is your job.

**What is given and working:** retrieval, the baseline RAG, graph building from a *pattern registry*, and the grounded answer step. The registry has **one pattern** (Invoice ID).

**What you do**
0. **Plug in your own LLM.** Add your key from any provider and, if needed, edit the one `call_llm` function so the notebook runs on it. Nothing else in the notebook touches the provider.
1. Explore the data and decide which other patterns identify a single event (**Kind 1**) and which are incidental fields reused across unrelated events (**Kind 2**: a room number, an amount, a semester...).
2. Add your patterns to the registry, each with a reason. List the lookalikes you *rejected*, and why.
3. Run the public question pack against baseline and grounded. Explain every outcome.
4. Say where the baseline fails and your graph works, and **one failure you still have.**

**No Neo4j?** If your team cannot use Neo4j Aura, replace the graph in sections 7 and 8 with lookup tables of your own (the Student Setup Guide shows the structure). Everything else, including the judging, is the same.

**What you submit:** this notebook, run top to bottom on a fresh runtime. The judges read your registry, your outcomes table and your remaining-failure note, then run a hidden question pack against your graph.

> Linking on the wrong thing is as bad as linking on nothing: a pattern that links unrelated records makes the system *invent* connections. Add a pattern only if you can say why two records sharing it are the same event.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't, then add `NEO4J_URI`, `NEO4J_USER` and `NEO4J_PASSWORD` to Colab's Secrets manager (padlock icon). If you hit `ServiceUnavailable`, the instance auto-paused: re-run this cell.

**Cannot use Neo4j?** Skip this cell. Replace the graph-building cell (section 7) and `graph_check` (section 8) with lookup-table versions that keep the same return shape. The Student Setup Guide shows the table structure.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset: Payment, shipment & invoice

138 records in 12 categories. An exporter's finance and logistics log: invoices, payments and disputes, collections, bookings, freight tracking, customs, tax and credit. Four pairs of categories deliberately share vocabulary. Most records stand alone; some carry an identifier or a code.

In [ ]:
DATASET_ID = "PAY"
DATASET_NAME = "Payment, shipment & invoice"

rec_categories = {
    'Invoicing': [
        'A sales invoice was generated after the goods were dispatched',
        'Invoice templates were updated to show the new tax registration',
        'A credit note was issued to correct an overbilled quantity',
        'Invoices for the month were batched and emailed to buyers',
        "An invoice was returned because the buyer's purchase reference was missing",
        'E-invoicing validation rejected a file with a formatting error',
        'A pro-forma invoice was sent ahead of a large order',
        'The billing team merged two invoices for the same buyer',
        'An invoice approval workflow was simplified for small amounts',
        'Invoice INV2044 was generated after the goods were dispatched to the buyer',
        'Invoice INV2031 was issued for a consignment of electronics',
        'Invoice INV2077 was raised against purchase order PO-4410',
        'Invoice INV2058 was issued for 8500 euros of electronic components',
        'Several invoices were held back until after the quarter-end settlement',
    ],
    'Payment_Processing': [
        'A bank transfer was received and matched to an open invoice',
        'Payment run was released after approvals were complete',
        'A wire transfer failed because of an incorrect account number',
        'Cheques deposited last week were cleared by the bank',
        'A partial payment was applied to the oldest outstanding balance',
        'Payment reminders were sent to buyers before the due date',
        'Direct debit mandates were refreshed for regular customers',
        'An overpayment was held in suspense pending instructions',
        'Payment confirmation was emailed to the supplier',
        'Invoice INV2044 was matched to a bank transfer and marked paid',
        'Payment PMT301 was released in the weekly payment run',
        'A refund of 8500 euros was processed to a customer after a cancelled order',
        'A refund of 12000 euros was issued after a billing error',
    ],
    'Payment_Disputes': [
        'A buyer disputed a charge for goods that arrived damaged',
        'The finance team requested proof of delivery from the carrier',
        'A short payment was explained by a pricing disagreement',
        'A dispute was escalated to the account manager',
        'A duplicate payment claim was investigated by the bank',
        'A buyer withheld payment pending a quality inspection',
        'The disputed amount was moved to a separate ledger',
        'A settlement was reached after a call between both finance teams',
        'A dispute log was reviewed in the weekly credit meeting',
        'Invoice INV2063 was disputed because part of the goods arrived damaged',
        'Payment PMT318 was withheld pending a quality inspection',
    ],
    'Collections': [
        'Overdue accounts were listed for follow-up calls',
        'A final demand letter was sent to a long-overdue buyer',
        'A payment plan was agreed for a struggling customer',
        'Collections escalated an old balance to legal counsel',
        'Dunning emails were sent in three stages',
        'A write-off proposal was prepared for irrecoverable balances',
        'Collections team confirmed a promise to pay by Friday',
        'Customer credit was suspended after repeated delays',
        'A collections report showed ageing by buyer',
        'Invoice INV2063 was moved to collections after the dispute stayed unresolved',
    ],
    'Shipment_Booking': [
        'A container booking was confirmed with the shipping line',
        'Cargo was consolidated into a shared container to save cost',
        'An air freight booking was made for urgent spare parts',
        'The booking cut-off was missed and the cargo was rolled to the next sailing',
        'A forwarder quoted rates for a full container load',
        'Packing lists were sent to the forwarder for the booking',
        'A booking amendment changed the delivery terms',
        'Dangerous goods documentation was added to a booking',
        'A shipper requested a refrigerated container for perishable goods',
        'Shipment SHP7115 was booked on the Friday sailing with the shipping line',
    ],
    'Freight_Tracking': [
        'A vessel delay pushed the arrival date by three days',
        'Tracking showed the container was loaded onto the ship',
        'A transshipment was completed at an intermediate hub',
        'The cargo arrived at the destination warehouse',
        'A tracking alert reported a stop at an unplanned location',
        'Proof of delivery was uploaded after the final mile',
        'The estimated arrival time was updated after a weather warning',
        'A container was reported damaged on arrival',
        'Live position updates were shared with the buyer',
        'Shipment SHP7101 was loaded onto the vessel at the origin port',
        'Shipment SHP7122 waited two days at Port Rotterdam for a berth',
        'Shipment SHP7130 was damaged by a crane accident at Port Rotterdam',
        'Electronics cargo was transshipped through Port Chennai',
        'A vessel was delayed by berth congestion at Port Chennai',
    ],
    'Customs_Clearance': [
        'A customs declaration was filed for imported machinery',
        'Customs held a container for document verification',
        'Duty was assessed on a consignment of electronics',
        'The broker requested the certificate of origin',
        'A customs inspection delayed the release of goods',
        'A tariff classification was corrected after a review',
        'Bonded warehouse storage was arranged for goods awaiting duty',
        'Import licence validity was checked before clearance',
        'A customs query was answered with the commercial invoice copy',
        'Shipment SHP7115 was held by customs for document verification',
        'Shipment SHP7101 was cleared after the duty was assessed',
    ],
    'Tax_Compliance': [
        'Quarterly VAT returns were prepared by the tax team',
        'A withholding tax certificate was issued to a supplier',
        'Tax authorities requested records of cross-border sales',
        'Input tax credit was reconciled with supplier filings',
        'A tax registration was updated after an address change',
        'Transfer pricing documentation was reviewed by advisors',
        'A refund claim for excess tax was submitted',
        'Tax documents were archived for seven years',
        'Tax rates were updated in the billing system',
        'Invoice INV2031 was included in the quarterly VAT return',
    ],
    'Supplier_Onboarding': [
        'A new supplier submitted registration documents for review',
        'Bank details of a supplier were verified by a test transfer',
        'Supplier due diligence checks were completed',
        'A supplier code of conduct was signed',
        'Onboarding of a logistics provider was approved by procurement',
        'A supplier was asked to update expired insurance certificates',
        'Supplier master data was cleaned of duplicate entries',
        'A trial order was placed with a newly onboarded supplier',
        'Supplier payment terms were agreed during onboarding',
        'Purchase order PO-4410 was placed with a newly onboarded supplier',
        'A supplier agreed to net 30 terms during onboarding',
    ],
    'FX_Treasury': [
        'A forward contract was booked to hedge a euro exposure',
        'Treasury reviewed the daily cash position across accounts',
        'Exchange rate gains were recorded for the month',
        'A currency conversion was delayed by a bank holiday',
        'Treasury moved surplus cash into a short-term deposit',
        'The hedging policy was updated for smaller exposures',
        'A bank charged a fee on a foreign currency transfer',
        'Cash forecasts were revised after a large receipt',
        'A credit line was drawn to cover a temporary gap',
        'Treasury brought the quarter-end settlement forward by two days',
        'Payment PMT325 was converted at the forward rate booked last month',
    ],
    'Audit_Reconciliation': [
        'Bank statements were reconciled with the ledger',
        'An audit sample checked invoice approvals against policy',
        'Unmatched receipts were investigated by the accounts team',
        'A reconciliation break was traced to a timing difference',
        'Intercompany balances were agreed between subsidiaries',
        'The auditor asked for evidence of shipment cut-off testing',
        'An old unreconciled item was cleared with approval',
        'Sub-ledger totals were tied back to the general ledger',
        'Reconciliation checklist was updated for the new system',
        'Payment PMT301 was matched to the bank statement during reconciliation',
    ],
    'Credit_Risk': [
        "A buyer's credit limit was reviewed after a payment delay",
        'Trade credit insurance was renewed for key accounts',
        'A credit check was run on a new customer',
        'Credit limit increase was declined pending financial statements',
        'A buyer was placed on hold after exceeding the limit',
        'The credit committee reviewed concentration by customer',
        'A bank guarantee was requested for a high-value order',
        'Credit scores were refreshed from the rating agency',
        'A letter of credit was examined for compliance',
        'A bank guarantee of 12000 euros was requested by the supplier',
        'A buyer asked to extend the net 30 terms after a payment delay',
        'A bank guarantee was requested for the high-value order PO-3871',
        'Invoice INV2058 triggered the credit hold after the buyer exceeded the limit',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. Explore: what could identify one event?

Read the records below. For each string with digits, ask: *if I saw this same string in a different record, would it still point at the same event?*

In [ ]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(rec_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

## 6. YOUR TASK: the pattern registry

Each entry becomes a graph node label. Two records that mention the *same identifier* get connected through that node. Edit this cell, then run it.

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Invoice',
        "regex": r"\bInvoice\s+(INV\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Invoice INV2044',
        "why": 'An invoice number names one invoice: its issue and its payment match are about the same document.',
    },
    # YOUR PATTERNS GO HERE. One entry per pattern, same five fields as above.
    # Not sure a pattern is right? Test it first:
    #   re.findall(r"your regex", campus_texts[i])   or loop over all campus_texts.
    # {
    #     "name": "Shipment",                      # becomes a node label in Neo4j
    #     "regex": r"...",                        # ONE capture group around the identifier
    #     "kind": 1,                              # 1 = names ONE event/person; 2 = incidental field
    #     "example": "...",                       # a string from the data it matches
    #     "why": "...",                           # why two records sharing it ARE the same event
    # },
]

REJECTED_CANDIDATES = [
    # Things that LOOK like identifiers but you decided NOT to link on. Credit is given for good reasons.
    # {"candidate": "...", "kind": 2, "why_rejected": "..."},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 7. Build the graph from your registry (given)

Re-run this cell every time you change the registry. The summary shows how many links each pattern created: a pattern with **0 shared identifiers** adds nothing, and one that links *many* records deserves suspicion.

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

run_write("MATCH (n) DETACH DELETE n")

# One :Event node per record
event_rows = [{"id": f"R{i}", "text": t, "category": c}
              for i, (t, c) in enumerate(zip(rec_texts, rec_labels))]
run_write("""
    UNWIND $rows AS row
    CREATE (e:Event {id: row.id, text: row.text, category: row.category})
""", rows=event_rows)

# One entity node per distinct identifier; records naming it are linked through it
by_label = {}
for i, text in enumerate(rec_texts):
    for name, ident in extract_entities(text):
        by_label.setdefault(name, []).append({"eid": f"R{i}", "xid": ident})

for name, rows in by_label.items():       # name was validated: letters/digits/underscore only
    run_write(f"""
        UNWIND $rows AS r
        MATCH (e:Event {{id: r.eid}})
        MERGE (x:{name} {{id: r.xid}})
        MERGE (e)-[:MENTIONS]->(x)
    """, rows=rows)

print("Graph built from your registry:")
for name, rows in by_label.items():
    ids = {}
    for r in rows:
        ids.setdefault(r["xid"], 0)
        ids[r["xid"]] += 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {len(rows):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

## 8. Grounded answer (given)

Retrieves, asks Neo4j (through `graph_check`) whether each pair of retrieved records is connected (and through which entity), then hands the LLM those facts with record IDs. `temperature=0`, so repeated runs give the same answer.

In [ ]:
def graph_check(id_a, id_b):
    """Asks Neo4j whether a path connects two records, and through which entities."""
    rows = run_read("""
        MATCH (a:Event {id: $a}), (b:Event {id: $b})
        OPTIONAL MATCH p = shortestPath((a)-[*..6]-(b))
        RETURN p IS NULL AS disconnected,
               [n IN nodes(p) WHERE NOT n:Event | labels(n)[0] + ' ' + n.id] AS via
    """, a=id_a, b=id_b)
    return rows[0] if rows else None


GROUNDED_PROMPT = """You answer questions about business-to-business trade operations (invoicing, payments, freight, customs, credit) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Amounts, port names, payment terms, settlement periods, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 9. The public question pack

8 adversarial questions (the right answer is *no recorded connection*) and 3 positive controls (the right answer is *yes*). The judges will run a **different, hidden** pack of the same shapes, so fix your **registry**, not individual questions.

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the partial payment applied to the oldest balance the same one explained by a '
              'pricing disagreement?',
  'trap_records': ['A partial payment was applied to the oldest outstanding balance',
                   'A short payment was explained by a pricing disagreement'],
  'correct_answer_must_state': 'No stated connection: neither record names an invoice or buyer, so '
                               'they cannot be shown to be the same payment.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (payment)'},
 {'id': 'P2',
  'question': 'Was the vessel delay about the container booked with the shipping line?',
  'trap_records': ['A container booking was confirmed with the shipping line',
                   'A vessel delay pushed the arrival date by three days'],
  'correct_answer_must_state': 'No stated connection: neither record names a shipment number.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (vessel)'},
 {'id': 'P3',
  'question': 'Was the container reported damaged on arrival the one held back by the customs '
              'inspection?',
  'trap_records': ['A customs inspection delayed the release of goods',
                   'A container was reported damaged on arrival'],
  'correct_answer_must_state': 'No stated connection: no shipment or container is named in both '
                               'records.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (customs)'},
 {'id': 'P4',
  'question': 'Were the dunning emails sent because of the payment reminders sent before the due '
              'date?',
  'trap_records': ['Dunning emails were sent in three stages',
                   'Payment reminders were sent to buyers before the due date'],
  'correct_answer_must_state': 'No stated connection: both are reminders, but they name no shared '
                               'invoice or buyer.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (reminders)'},
 {'id': 'P5',
  'question': 'Did the transshipment of cargo through Port Chennai cause the vessel delay by berth '
              'congestion there?',
  'trap_records': ['Electronics cargo was transshipped through Port Chennai',
                   'A vessel was delayed by berth congestion at Port Chennai'],
  'correct_answer_must_state': 'No stated connection: both mention Port Chennai, but a port is a '
                               'place used by many shipments, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: port'},
 {'id': 'P6',
  'question': "Was the buyer's request to extend the net 30 terms linked to the supplier agreeing "
              'to net 30 terms in onboarding?',
  'trap_records': ['A supplier agreed to net 30 terms during onboarding',
                   'A buyer asked to extend the net 30 terms after a payment delay'],
  'correct_answer_must_state': 'No stated connection: net 30 is a standard payment term shared by '
                               'many parties.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: payment terms'},
 {'id': 'P7',
  'question': 'Was the 12000 euro refund covered by the 12000 euro bank guarantee?',
  'trap_records': ['A refund of 12000 euros was issued after a billing error',
                   'A bank guarantee of 12000 euros was requested by the supplier'],
  'correct_answer_must_state': 'No stated connection: the amounts match by coincidence, and the '
                               'records name no shared invoice or order.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: amount'},
 {'id': 'P8',
  'question': 'Was the customs query answered with the invoice copy because supplier master data '
              'was cleaned?',
  'trap_records': ['A customs query was answered with the commercial invoice copy',
                   'Supplier master data was cleaned of duplicate entries'],
  'correct_answer_must_state': 'No stated connection between a customs query and a master data '
                               'clean-up; any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Was Invoice INV2044 generated after dispatch and later matched to a bank transfer?',
  'shared_identifier': 'Invoice INV2044',
  'shared_records': ['Invoice INV2044 was generated after the goods were dispatched to the buyer',
                     'Invoice INV2044 was matched to a bank transfer and marked paid'],
  'correct_answer_must_state': 'Yes -- both records name Invoice INV2044.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Was Invoice INV2063 disputed for damaged goods and then moved to collections?',
  'shared_identifier': 'Invoice INV2063',
  'shared_records': ['Invoice INV2063 was disputed because part of the goods arrived damaged',
                     'Invoice INV2063 was moved to collections after the dispute stayed '
                     'unresolved'],
  'correct_answer_must_state': 'Yes -- both records name Invoice INV2063.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Shipment SHP7115 booked on the Friday sailing and later held by customs?',
  'shared_identifier': 'Shipment SHP7115',
  'shared_records': ['Shipment SHP7115 was booked on the Friday sailing with the shipping line',
                     'Shipment SHP7115 was held by customs for document verification'],
  'correct_answer_must_state': 'Yes -- both records name Shipment SHP7115.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### Adversarial questions

In [ ]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)

### Positive controls: can your agent say *yes*?

In [ ]:
pos_rows = run_pack(PUBLIC_POSITIVE)

## 10. YOUR TASK: explain the outcomes

For every public question, record whether the **baseline fabricated** a connection (read its answer, `True`/`False`), whether your **grounded** answer was correct, and **why**, in terms of your patterns. Then name one failure you still have. Be specific: a short honest note beats a vague claim.

In [ ]:
OUTCOMES = [
    # {"id": "P1", "baseline_fabricated": None, "grounded_correct": None, "explanation": "..."},
    # one entry for each of P1-P8 and PP1-PP3
]

REMAINING_FAILURE = """
TODO: one case, or kind of case, where your system still gets it wrong (or could), and why.
"""

## 11. Submission summary

Run this last. This is what the judges read first.

In [ ]:
import json

def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    # one machine-readable line the judges' tool reads: keep it
    print("REGISTRY_JSON:", json.dumps([{"name": p["name"], "regex": p["regex"]} for p in PATTERNS]))
    print("DATASET_ID:", DATASET_ID)
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()

## 12. Before you submit

- Restart the runtime and **Run all**: it must work from a fresh state.
- `validate_registry()` reports no problems.
- `OUTCOMES` has an entry for every public question; `REMAINING_FAILURE` is filled in.
- Submit by the deadline announced on Day 1.